Nama  : Roland Albertian Sehapikang<br>
Kelas : IF403<br>
NIM   : 240401010294

In [1]:
import pandas as pd
import numpy as np
from scipy.stats.mstats import winsorize
import requests
from pandas import json_normalize

df = pd.read_csv('https://raw.githubusercontent.com/Ganzer13/DataScience_240401010294_Roland/main/housing_dirty.csv')

print('Shape awal:', df.shape)
print('\n--- Info Dataset ---')
df.info()
print('\n--- Statistik Deskriptif ---')
print(df.describe())
print('\n--- Jumlah Missing Values ---')
print(df.isnull().sum())

Shape awal: (130, 7)

--- Info Dataset ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 130 entries, 0 to 129
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            130 non-null    int64  
 1   luas_m2       112 non-null    float64
 2   harga_juta    113 non-null    float64
 3   kota          130 non-null    object 
 4   kamar         120 non-null    float64
 5   tahun_bangun  130 non-null    int64  
 6   kondisi       130 non-null    object 
dtypes: float64(3), int64(2), object(2)
memory usage: 7.2+ KB

--- Statistik Deskriptif ---
               id      luas_m2    harga_juta       kamar  tahun_bangun
count  130.000000   112.000000  1.130000e+02  120.000000    130.000000
mean    65.500000   267.627679  8.856325e+05    3.433333   2062.638462
std     37.671829   885.664181  9.407144e+06    1.776283    701.684043
min      1.000000   -50.000000 -5.000000e+02    1.000000   1890.000000
25%     33.250000  

**Temuan:** Pada tahap awal, kita melihat struktur asli dari dataset housing_dirty.csv yang memiliki 130 baris dan 7 kolom. Terdapat missing values pada beberapa kolom seperti luas_m2, harga_juta, dan kamar, serta terindikasi adanya baris duplikat maupun masalah konsistensi string.

In [2]:
df.drop_duplicates(inplace=True)
print('Setelah hapus duplikat:', df.shape)

df['kota'] = df['kota'].str.strip().str.title()
df['kondisi'] = df['kondisi'].str.strip().str.lower()

print('\nNilai unik pada kolom kota setelah normalisasi:')
print(df['kota'].unique())

Setelah hapus duplikat: (130, 7)

Nilai unik pada kolom kota setelah normalisasi:
['Jogja' 'Medan' 'Depok' 'Ygy' 'Jakarta' 'Yogyakarta' 'Bandung' 'Surabaya'
 'Dpk' 'Sby' 'Makassar' 'Mdn' 'Semarang' 'Smg' 'Bdg' 'Mksr']


**Temuan:** Fungsi drop_duplicates(inplace=True) digunakan untuk menghapus baris-baris observasi properti yang tercatat lebih dari satu kali. Untuk mengatasi inkonsistensi string (seperti 'jakarta' vs 'Jakarta'), kita menggunakan str.strip() untuk menghapus spasi ekstra dan str.title() / str.lower() untuk menyamakan format kapitalisasi huruf.

In [3]:
df['luas_m2'] = df['luas_m2'].fillna(df['luas_m2'].median())
df['harga_juta'] = df['harga_juta'].fillna(df['harga_juta'].median())

df['kamar'] = df['kamar'].fillna(df['kamar'].mode()[0])

for col in ['harga_juta', 'luas_m2', 'tahun_bangun']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    df[col] = df[col].clip(lower=Q1 - 1.5 * IQR, upper=Q3 + 1.5 * IQR)

print('\nJumlah Missing Values setelah imputasi:')
print(df.isnull().sum())


Jumlah Missing Values setelah imputasi:
id              0
luas_m2         0
harga_juta      0
kota            0
kamar           0
tahun_bangun    0
kondisi         0
dtype: int64


**Temuan:** Missing values pada kolom numerik (luas_m2, harga_juta) diisi menggunakan median karena nilai rata-rata (mean) rentan terdistorsi oleh data kotor, sementara kolom kamar diisi menggunakan nilai modus. Outlier seperti harga tidak wajar atau tahun bangun di bawah 1950 ditangani dengan teknik capping atau Winsorization menggunakan metode clip() berdasarkan batas pagar IQR (Interquartile Range).

In [4]:
assert df.isnull().sum().sum() == 0, 'Masih ada missing!'
assert df.duplicated().sum() == 0, 'Masih ada duplikat!'

print('Shape akhir dataset bersih:', df.shape)

df.to_csv('housing_clean.csv', index=False)
print('Dataset bersih berhasil tersimpan sebagai housing_clean.csv!')

Shape akhir dataset bersih: (130, 7)
Dataset bersih berhasil tersimpan sebagai housing_clean.csv!


**Temuan:** Penggunaan perintah assert merupakan praktik yang baik (best practice) untuk memastikan validasi akhir berjalan otomatis. Dataset yang sudah sepenuhnya bebas dari missing values, baris duplikat, dan outliers akhirnya diekspor ke file housing_clean.csv untuk keperluan analisis atau pemodelan selanjutnya.

In [5]:
URL = "https://jsonplaceholder.typicode.com/users"

response = requests.get(URL, timeout=10)

if response.status_code == 200:
    data = response.json()

    df_api = json_normalize(data, sep='_')

    print("Data API berhasil diekstrak!")

    display(df_api[['id', 'name', 'email', 'address_city']].head())
else:
    print(f'Error saat mengakses API: {response.status_code}')

Data API berhasil diekstrak!


,id,name,email,address_city
0,1,Leanne Graham,Sincere@april.biz,Gwenborough
1,2,Ervin Howell,Shanna@melissa.tv,Wisokyburgh
2,3,Clementine Bauch,Nathan@yesenia.net,McKenziehaven
3,4,Patricia Lebsack,Julianne.OConner@kory.org,South Elvis
4,5,Chelsey Dietrich,Lucio_Hettinger@annie.ca,Roscoeview


**Temuan:** Menggunakan modul requests, kita berhasil menarik data dari REST API JSONPlaceholder. Penggunaan fungsi json_normalize dari Pandas sangat memudahkan dalam mengubah format JSON bertingkat (nested array dari API) ke dalam bentuk tabel tabular (DataFrame) datar (flat). Selalu menambahkan timeout=10 dan memverifikasi status_code == 200 merupakan standar yang aman saat berinteraksi dengan API eksternal.

# Kesimpulan Singkat:

**1. Apa yang dipelajari?**
* **Pembersihan Data Dasar:** Cara mendeteksi dan menghapus baris duplikat menggunakan .drop_duplicates(), serta menormalisasi format teks (huruf besar/kecil dan spasi berlebih) menggunakan manipulasi string .str.strip(), .str.title(), dan .str.lower().
* **Imputasi Missing Values:** Strategi mengisi data yang kosong, yaitu menggunakan nilai median untuk data numerik (karena lebih kebal terhadap outlier) dan modus untuk data diskret/kategorik.

* **Penanganan Outlier:** Menggunakan metode pagar IQR (Interquartile Range) yang dikombinasikan dengan fungsi .clip() dari Pandas untuk membatasi (capping) nilai ekstrem agar sesuai dengan batas atas dan batas bawah.

* **Validasi Otomatis:** Menggunakan fungsi assert untuk memastikan tidak ada lagi data kosong atau duplikat sebelum dataset diekspor ke file CSV baru.

* **Ekstraksi Data API:** Mempelajari cara mengambil data dari web melalui REST API menggunakan pustaka requests, dan meratakan (flattening) struktur data JSON yang bersarang menjadi tabel DataFrame yang rapi menggunakan json_normalize().

**2. Temuan Utama**
* **Karakteristik Data Mentah:** Data di dunia nyata (seperti housing_dirty.csv) hampir selalu kotor. Masalah seperti salah ketik (inkonsistensi kapitalisasi), data ganda, dan kolom kosong adalah hal yang wajar dan harus ditangani sebelum tahap pemodelan.

* **Pentingnya Statistik Robust:** Dalam menangani data harga dan luas properti yang memiliki nilai ekstrem (outlier), penggunaan median terbukti lebih aman dan representatif dibandingkan mean (rata-rata).

* **Integrasi Data Eksternal:** Data tidak hanya berasal dari file lokal (CSV), tetapi juga bisa ditarik secara langsung dan dinamis dari internet (API) ke dalam notebook analitik.

**3. Keterbatasan atau Pertanyaan yang muncul**
* **Keterbatasan Analisis:** * Mengisi nilai kosong dengan median atau modus secara global (satu nilai untuk seluruh baris yang kosong) adalah metode dasar yang bisa mengabaikan korelasi antar-variabel (misalnya, harga rumah seharusnya bergantung pada luasnya).
  * Pemotongan outlier dengan .clip() bisa menghilangkan variasi data yang valid (misalnya, jika rumah tersebut memang benar-benar rumah mewah yang sangat besar dan mahal).
* **Pertanyaan yang Muncul:**
  * Bagaimana cara menggunakan metode imputasi yang lebih cerdas dan berbasis Machine Learning (seperti KNN Imputer) agar pengisian data yang hilang lebih akurat?

  * Bagaimana cara mengekstrak data dari API yang tidak bersifat publik, melainkan membutuhkan login atau token autentikasi (seperti API berbayar)?
